Défi quotidien : Gestion et analyse des données en Python


Ce que tu apprendras
Techniques avancées de normalisation, réduction et agrégation des données.
Compétences dans la collecte, l’exploration, l’intégration et le nettoyage de données avec Python.
Maîtrise de l’utilisation de Pandas pour la manipulation complexe de données.


Votre tâche
Téléchargez et importez le jeu de données Data Science Job Salary Dataset.
Normalisez la colonne « salaire » en utilisant la normalisation Min-Max qui scale toutes les valeurs salariales entre 0 et 1.
Mettez en œuvre une réduction de dimensionnalité comme l’analyse des composantes principales (PCA) ou t-SNE pour diminuer le nombre de caractéristiques (colonnes) dans l’ensemble de données.
Regroupez l’ensemble de données par la colonne « experience_level » et calculez le salaire moyen et médian pour chaque niveau d’expérience (par exemple, Junior, Mid-level, Senior).
Indice :
Pour rappel, la normalisation est cruciale lorsqu’on traite des données ayant des plages différentes. Par exemple, les données sur les salaires peuvent avoir une large gamme (par exemple, de 20 000 $ à 200 000 $). En mettant à l’échelle les données grâce à la normalisation Min-Max, vous vous assurez que toutes les valeurs salariales restent dans une fourchette cohérente (0 à 1). C’est particulièrement utile lorsque les données sont utilisées dans des modèles d’apprentissage automatique, car certains algorithmes (comme k-plus proches voisins ou réseaux de neurones) fonctionnent mieux lorsque les caractéristiques sont normalisées. Cela garantit qu’aucun salaire unique ne domine le processus d’apprentissage, rendant l’analyse plus équilibrée.

La réduction de la dimensionnalité aide à simplifier les ensembles de données complexes en réduisant le nombre de variables prises en compte. Cela peut rendre les données plus gérables et aider à éviter la malédiction de la dimensionnalité — un phénomène où les modèles d’apprentissage automatique peinent lorsqu’ils manipulent des données de haute dimension.
L’ACP, par exemple, aide à conserver les informations les plus importantes (la variance) du jeu de données tout en réduisant le bruit et la redondance.
Cela peut également accélérer le processus d’entraînement des modèles et aider à visualiser les données en moins de dimensions.

L’agrégation des données aide à comprendre les tendances au sein des sous-groupes de données de l’ensemble.
Calculer les salaires moyens et médians pour chaque niveau d’expérience donne un aperçu de la répartition de la rémunération et des disparités selon les différents niveaux d’emploi. Ce type d’agrégation peut aider à répondre à des questions métier telles que « Comment le salaire évolue-t-il avec l’expérience ? » ou « Quelle est la répartition des salaires pour les postes de niveau supérieur ? »



Soumettez votre défi quotidien
Téléchargez vos scripts Python, vos visualisations et un résumé de vos connaissances sur GitHub.

Une dernière chose : bonne chance !

In [8]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ==========================================
# 1. Importation et Nettoyage de la structure
# ==========================================
print("Chargement des données...")
# Mise à jour du nom du fichier ici :
df = pd.read_csv('datascience_salaries.csv')

# On supprime les espaces invisibles autour des noms de colonnes
df.columns = df.columns.str.strip()

# ==========================================
# 2. Normalisation Min-Max
# ==========================================
# Détection automatique de la colonne de salaire disponible
col_salaire = 'salary_in_usd' if 'salary_in_usd' in df.columns else 'salary'

scaler = MinMaxScaler()
df['salary_normalized'] = scaler.fit_transform(df[[col_salaire]])

print(f"\nAperçu des salaires ({col_salaire}) normalisés :")
print(df[[col_salaire, 'salary_normalized']].head())

# ==========================================
# 3. Réduction de Dimensionnalité (PCA)
# ==========================================
# On liste les colonnes idéales, mais on ne garde QUE celles 
# qui existent vraiment dans ton DataFrame pour éviter l'erreur KeyError.
colonnes_potentielles = ['experience_level', 'employment_type', 'job_title', 'company_size']
features_to_encode = [col for col in colonnes_potentielles if col in df.columns]

print(f"\nColonnes utilisées pour l'ACP : {features_to_encode}")

# Encodage One-Hot des colonnes catégorielles existantes
df_features = pd.get_dummies(df[features_to_encode], drop_first=True)

# Ajout du salaire normalisé
df_features['salary_normalized'] = df['salary_normalized']

# Application de l'ACP (2 composantes)
n_comp = min(2, df_features.shape[1]) 
pca = PCA(n_components=n_comp)
pca_result = pca.fit_transform(df_features)

# Ajout dynamique des composantes au DataFrame
for i in range(n_comp):
    df[f'PCA_Component_{i+1}'] = pca_result[:, i]

print(f"Variance expliquée par l'ACP : {pca.explained_variance_ratio_.sum() * 100:.2f}%")

# ==========================================
# 4. Agrégation des données
# ==========================================
# On vérifie que la colonne 'experience_level' existe bien avant de grouper
if 'experience_level' in df.columns:
    salary_stats = df.groupby('experience_level')[col_salaire].agg(['mean', 'median']).reset_index()

    # Renommer les colonnes pour l'affichage
    salary_stats.rename(columns={
        'experience_level': 'Niveau d\'expérience',
        'mean': 'Salaire Moyen',
        'median': 'Salaire Médian'
    }, inplace=True)

    # Arrondir pour la lisibilité
    salary_stats['Salaire Moyen'] = salary_stats['Salaire Moyen'].round(2)
    salary_stats['Salaire Médian'] = salary_stats['Salaire Médian'].round(2)

    print("\n--- Statistiques des Salaires par Niveau d'Expérience ---")
    print(salary_stats.to_string(index=False))
else:
    print("\nLa colonne 'experience_level' est absente. Impossible d'effectuer l'agrégation demandée.")

Chargement des données...

Aperçu des salaires (salary) normalisés :
   salary  salary_normalized
0  149000           0.601010
1  120000           0.454545
2   68000           0.191919
3  120000           0.454545
4  149000           0.601010

Colonnes utilisées pour l'ACP : ['experience_level', 'job_title']
Variance expliquée par l'ACP : 64.12%

--- Statistiques des Salaires par Niveau d'Expérience ---
Niveau d'expérience  Salaire Moyen  Salaire Médian
              Entry       36111.11         30000.0
          Executive       76076.92         46000.0
                Mid       51786.89         51000.0
             Senior       75088.03         68000.0


Normalisation (MinMaxScaler) : Les modèles basés sur les distances (comme les KNN ou les réseaux de neurones) sont très sensibles à l'échelle des données. Réduire les salaires sur une échelle de 0 à 1 empêche cette caractéristique d'écraser les autres variables lors de l'entraînement du modèle, tout en conservant la distribution exacte des données.

Réduction de dimensionnalité (PCA) : Après avoir encodé les variables catégorielles (comme les titres de poste ou la taille de l'entreprise), le jeu de données s'est considérablement élargi (malédiction de la dimensionnalité). L'ACP a permis de compresser ces informations en composantes principales, réduisant le bruit et facilitant une potentielle visualisation en 2D, tout en conservant un maximum de variance.

Agrégation (Groupby) : L'analyse groupée confirme la logique métier : l'agrégation mean et median montre l'évolution de la rémunération. L'utilisation conjointe de la moyenne et de la médiane est cruciale, car la moyenne peut être biaisée par quelques salaires extrêmes (outliers), tandis que la médiane reflète le salaire "typique" du groupe.